# The likelihood route: a measurement as an in-graph soft constraint

The prior route moves the amplitude's prior. The likelihood route (decision D6.3) adds a term to
the likelihood instead. A `Measurement` says "estimand `E` was measured at `estimate ± se`"; the
surface says what `E` *is* as a function of its parameters. `constraint_for` builds `E(θ)` as an
expression over the surface's mean tree — `estimands.estimand_expr` gives the per-cell contrast,
the experiment's dose paths replace the dose columns, `Reduce` nodes aggregate over the window
and the units to the estimand's functional (decision 0002.20) — and wraps it in a
`core.Constraint`:

    log p(panel | θ) + log N(estimate | E(θ), se) + log p(θ)

Because `E` depends on the amplitude *and* the curve shape, the measurement pulls every
parameter it depends on. That is what distinguishes this route from the prior route.

Aggregation rule (0002.20): a *contrast* is summed over the window's periods for a `cumulative`
basis and averaged for `per_period`, then averaged over units at the `individual` level and
summed at `cluster` / `aggregate`. A *ratio* or a *marginal* is the plain mean of the per-cell
quantity. `area`, `elasticity`, a marginal through carryover, strata weights and conditioning are
a typed `Unsupported` pointing at `estimands.realize`.

In [ ]:
import numpy as np

from axiom.calibrate import (
    Agreement, AgreementVerdict, Measurement, agreement, attach, constraint_for, fit_calibrated,
    lognormal_mu_from_moments, lognormal_sigma_from_moments,
)
from axiom.core import (
    Constraint, Intervention, ModelSpec, Population, Posterior, TimeWindow, Unsupported,
    log_density, unconstrain, value,
)
from axiom.estimands import Estimand, Level, Quantity
from axiom.sim import surface_world
from axiom.surface import HillKernel, fit

from axiom.display import enable

enable();  # every axiom result renders itself from here on

## Log-scale moments

When the estimand's `quantity.scale` is `"log"`, the measurement's `se` is read as the
uncertainty of a positive quantity on the multiplicative scale and the constraint becomes a
lognormal with `scale = lognormal_sigma_from_moments(estimate, se) = sqrt(log(1 + (se/estimate)²))`.
`lognormal_mu_from_moments` is the matching location `log(estimate) − sigma²/2`.

In [ ]:
sigma = lognormal_sigma_from_moments(2.0, 0.3)
mu = lognormal_mu_from_moments(2.0, 0.3)
print(f"sigma={sigma:.5f} mu={mu:.5f}")
print("golden (1.0, 0.5) ->", lognormal_sigma_from_moments(1.0, 0.5))
print("moments round-trip:", round(float(np.exp(mu + sigma**2 / 2)), 6),
      round(float(np.sqrt((np.exp(sigma**2) - 1) * np.exp(2 * mu + sigma**2))), 6))

## A world and a measurement

Same setup as the prior-route notebook: a one-treatment Hill surface, a noisy observational
panel, and a precise simulated experiment on the per-unit cumulative contrast between dose 2 and
dose 0.

In [ ]:
TRUTH = {"beta_a": 1.5, "k_a": 1.0, "s_a": 2.0}
world = surface_world(
    n_units=4, n_periods=12, treatments=("a",), kernels=HillKernel(reference_dose=1.0),
    intercept="shared", truth=TRUTH, noise_sd=0.6, seed=7,
)
spec, surface, data = world.spec, world.surface, world.data


def estimand(kind="contrast", hi=2.0, lo=0.0, window=None, level="individual", scale="natural", dim=None):
    return Estimand(
        name=f"lift_{kind}",
        quantity=Quantity(kind=kind, scale=scale),
        treatment=spec.treatment("a"),
        intervention=Intervention(doses={"a": hi}),
        reference=Intervention(doses={"a": lo}) if kind in ("contrast", "ratio", "area") else None,
        outcome=spec.outcome,
        population=Population(name="panel_units"),
        window=window or TimeWindow(start=0, stop=world.n_periods, basis="cumulative"),
        level=Level(unit=level),
        dimension=dim or spec.outcome_dimension,
    )


lift = estimand()
truth_contrast = float(np.mean(np.sum(world.forward({"a": 2.0}) - world.forward({"a": 0.0}), axis=1)))
se_exp = 0.05 * truth_contrast
experiment = Measurement(estimand=lift, estimate=truth_contrast * 1.01, se=se_exp,
                         method="randomized_holdout", n_units=40, n_periods=12, source="holdout-2026Q1")
print(f"truth {truth_contrast:.4f}; measured {experiment.estimate:.4f} ± {experiment.se:.4f}")

## `constraint_for` builds the expression; evaluated at the truth it equals the estimand

The constraint's `expr` is a scalar tree over the surface's parameters and the panel's data.
Evaluating it at the world's true parameters through `core.value` reproduces the true contrast —
the same arithmetic the likelihood uses.

In [ ]:
c = constraint_for(experiment, surface, data)
assert isinstance(c, Constraint)
print(c.name, "| family", c.family, "| observed", round(c.observed, 4), "| scale", round(c.scale, 4))
print("E(theta_true) =", round(float(value(c.expr, data=data, params=world.theta)), 4))
print({k: v for k, v in c.detail.items() if k in ("quantity", "doses", "window", "level", "n_units", "n_periods")})

The experiment's dose paths can replace the panel's observed doses (`doses=`); a per-period
window and a `cluster` level change the aggregation; a log-scale estimand becomes a lognormal
constraint; and the cases the route cannot express return `Unsupported` with a reason.

In [ ]:
per_period = constraint_for(Measurement(estimand=estimand(window=TimeWindow(start=2, stop=8, basis="per_period"), level="cluster"),
                                        estimate=1.0, se=0.1, source="s2"), surface, data)
print("per-period cluster-level contrast at truth:", round(float(value(per_period.expr, data=data, params=world.theta)), 4),
      "| window", per_period.detail["window"])

ratio_log = constraint_for(Measurement(estimand=estimand(kind="ratio", lo=0.5, scale="log", dim=spec.outcome_dimension / spec.dose_dimension("a")),
                                       estimate=0.8, se=0.1, source="s3"), surface, data, doses={"a": np.linspace(0.2, 1.0, 12)})
print("log-scale ratio ->", ratio_log.family, "scale", round(ratio_log.scale, 5), "| doses", ratio_log.detail["doses"])

area = constraint_for(Measurement(estimand=estimand(kind="area", dim=spec.outcome_dimension * spec.dose_dimension("a")),
                                  estimate=1.0, se=0.1, source="s4"), surface, data)
print(type(area).__name__, "->", area.reason[:90], "...")

## `attach`: the surface's `ModelSpec` plus the constraint

`attach` returns `surface.model` with the constraint appended. The mean is unchanged; the log
density at any point differs from the unconstrained one by exactly `log N(estimate | E(θ), se)`.

In [ ]:
from scipy import stats

model = attach(experiment, surface, data)
assert isinstance(model, ModelSpec)
print("same mean:", model.mean == surface.model.mean, "| constraints:", [k.name for k in model.constraints])
z = unconstrain(model, world.theta)
delta = log_density(model, data, z) - log_density(surface.model, data, z)
E = float(value(c.expr, data=data, params=world.theta))
print("log-density difference:", round(delta, 6), "= log N(obs | E, se):", round(float(stats.norm.logpdf(c.observed, E, c.scale)), 6))

## `fit_calibrated`: `surface.fit` on the constrained model

The same steps as `surface.fit` run on the constrained `ModelSpec`. The returned `FitResult`
holds the unconstrained `Surface` (predictions use the mean; the constraint only shapes the
posterior) and records every constraint in `provenance`. Compared with the plain fit, the
amplitude posterior tightens and moves toward the truth. The shape `s_a` and scale `k_a` are
pulled too — directly, because the constraint's expression `E(θ)` depends on them, not merely
through a posterior correlation with the amplitude as in the prior route.

In [ ]:
plain = fit(spec, world.panel, backend="laplace", draws=500, seed=0)
calibrated = fit_calibrated(spec, world.panel, [experiment], backend="laplace", draws=500, seed=0)
assert not isinstance(calibrated, Unsupported)
assert isinstance(plain.posterior, Posterior) and isinstance(calibrated.posterior, Posterior)
for name in ("beta_a", "s_a", "k_a"):
    a, b = plain.posterior.flat(name), calibrated.posterior.flat(name)
    print(f"{name}: plain {a.mean():.3f} ± {a.std():.3f}   calibrated {b.mean():.3f} ± {b.std():.3f}   truth {TRUTH[name]}")
print("route:", calibrated.provenance["route"], "| constraints:", calibrated.provenance["constraints"][0]["name"],
      "| measurement sources:", calibrated.provenance["measurement_sources"])

## Did it take? `agreement`

`agreement` realizes the measurement's estimand on the fitted producer (`estimands.realize`, the
same arithmetic every estimand goes through) and compares. With the measurement `m ± se` and the
posterior mean `μ` and sd `τ` of the realized estimand,

    z = (m − μ) / sqrt(se² + τ²),    p = 2·(1 − Φ(|z|))

`inside` asks a different question — whether the measurement's point lies in the posterior
interval — so both are reported. The `AgreementVerdict` bands are explicit arguments:
`|z| < tension_at` (default 1) is `agrees`, below `disagrees_at` (default 2) is `tension`, else
`disagrees`.

In [ ]:
ok = agreement(calibrated, experiment, definition="hdi", mass=0.9, seed=0)
assert isinstance(ok, Agreement)
verdict: AgreementVerdict = ok.verdict
print(f"calibrated fit: z={ok.z:+.3f} p={ok.p:.3f} inside={ok.inside} verdict={verdict}")
print(f"  posterior {ok.posterior_mean:.4f} ± {ok.posterior_sd:.4f}; interval {ok.interval}")
print("  realized status:", ok.realized.status, "| assumptions:", [a.name for a in ok.realized.assumptions])

before = agreement(plain, experiment)
print(f"plain fit:      z={before.z:+.3f} p={before.p:.3f} inside={before.inside} verdict={before.verdict}")

far = agreement(calibrated, Measurement(estimand=lift, estimate=truth_contrast * 1.5, se=se_exp, source="implausible"))
print(f"an implausible measurement: z={far.z:+.2f} verdict={far.verdict}")